In [ ]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"


df = pd.read_csv(r"C:\Users\purav\OneDrive\Desktop\Internship\telehealth_patients.csv")

def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        
        active_patients=(
            group["retention_status"].astype(str).str.lower().eq("active").sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(
                group["treatment_duration_weeks"].mean(),2
            ),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result
    
tools = [
    {
        "name": "get_df_info",
        "description": "Returns dataframe metadata including columns, row count, data types and first five rows.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Performs cohort analysis on the dataframe, grouping by program type and calculating metrics such as number of patients, average treatment duration, active and churned patients, and retention rate.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]









def add_user_message(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_message(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat(messages, system=None, temperature=0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "tools": tools,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    return client.messages.create(**params)


def text_from_message(message):
    texts = []

    for block in message.content:
        if block.type == "text":
            texts.append(block.text)

    return "\n".join(texts)


def run_tool(tool_name, tool_input):
    if tool_name == "get_df_info":
        return get_df_info()
    elif tool_name =='run_cohort_analysis':
        return run_cohort_analysis

    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []

    for block in message.content:

        if block.type != "tool_use":
            continue

        try:
            result = run_tool(block.name, block.input)

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        except Exception as e:
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                }
            )

    return tool_results

messages = [
    {
        "role": "user",
        "content": "Tell me about the dataframe."
    }
]

response = chat(messages)

if response.stop_reason == "tool_use":

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    tool_results = run_tools(response)

    messages.append(
        {
            "role": "user",
            "content": tool_results,
        }
    )

    response = chat(messages)

print(text_from_message(response))

Here's a summary of the dataframe:

**Overview:**
- **Total Rows:** 300 patients
- **Total Columns:** 6

**Columns and Data Types:**
1. **patient_id** (object/string) - Unique patient identifier (e.g., "PT-0001")
2. **program_type** (object/string) - Type of treatment program (e.g., "Peptides", "Testosterone")
3. **start_date** (object/string) - Program start date in DD-MM-YYYY format
4. **treatment_duration_weeks** (int64) - Duration of treatment in weeks (numeric)
5. **retention_status** (object/string) - Status of patient retention (e.g., "Active", "Completed")
6. **drop_off_reason** (object/string) - Reason for dropping off the program (e.g., "N/A" for active/completed patients)

**Sample Data:**
The first 5 rows show a mix of:
- Patients on "Peptides" and "Testosterone" programs
- Treatment durations ranging from 14 to 52 weeks
- Retention statuses including "Active" and "Completed"
- Most entries show "N/A" for drop_off_reason (likely because those patients are still active or co